In [ ]:
import os
hsaDir = os.path.abspath("../../")

In [ ]:
from heliostat_surface_analysis.loader.txt_loader import TXTLoader
from heliostat_surface_analysis.preprocessor.basic_preprocessor import BasicPreprocessor
from heliostat_surface_analysis.mesher.delaunay_mesher import DelaunayMesher
from heliostat_surface_analysis.analyser.fit_surface_analyser import FitSurfaceAnalyser
from heliostat_surface_analysis.aggregator.aggregator import WeightedMLEAggregator
from heliostat_surface_analysis.analysis_manager import AnalysisManager
from heliostat_surface_analysis.plotting.plotting import plot_2d_surface,plot_3d_surface,set_plot_font_size

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
%matplotlib inline

# Setup analysis components

In [ ]:
preprocessor = BasicPreprocessor()
mesher = DelaunayMesher({'max_aspect_ratio':10,'verbose':True})
analyser = FitSurfaceAnalyser({"surface":"circular_paraboloid"})
aggregator = WeightedMLEAggregator()

In [ ]:
#caseName = "CSIRO Solar Field 1 heliostats"
caseName = "CSIRO Solar Field 2 heliostats"

In [ ]:
if caseName=="CSIRO Solar Field 1 heliostats":
    caseDir = os.path.join(hsaDir,"examples","circular_paraboloids","CSIRO_SolarField1_heliostats")
    loader = TXTLoader({"units":"m"})
elif caseName=="CSIRO Solar Field 2 heliostats":
    caseDir = os.path.join(hsaDir,"examples","circular_paraboloids","CSIRO_SolarField2_heliostats")
    loader = TXTLoader({"units":"mm"})
else:
    raise RuntimeError("caseName: %s not recognised"%caseName)
scansDir = os.path.join(caseDir,"scans")
scanFiles = sorted(os.listdir(scansDir))
print(len(scanFiles))

# Analyse a random scan

In [ ]:
scanFile = random.choice(scanFiles)
scanPath = os.path.join(scansDir,scanFile)

## 1. via the analyse_scan function (quick and easy but not very flexible)

In [ ]:
result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=True,write_stls=False)
f = 1/4./result["coeff: a(x^2 + y^2)"]
print("f = %0.1fm, sigma = %0.2fmrad"%(f,result["aggregate"]*1e3))

## 2. via custom code (allows modification of figure size and data range etc)

In [ ]:
scan_name = os.path.splitext(os.path.basename(scanPath))[0]
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
mesh = mesher.create_mesh(x, y, z)
areas, normals, coeffs, ideal_normals, ideal_zs, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
sigma = aggregator.aggregate(slope_errors,areas)

f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.set_title("%s (%0.2f mrad)"%(scan_name,sigma*1e3))
plot_2d_surface(mesh,slope_errors,label="slope error (rad)",ax=ax,fontsize=16,cbxy=(4.0,0.29))
f,ax = plt.subplots()
f.set_size_inches(10,6)
ax.set_title("%s (%0.2f mrad)"%(scan_name,sigma*1e3))
ax.hist(slope_errors,bins=20,rwidth=0.85)
ax.grid()
ax.set_xlabel("slope error (rad)")
set_plot_font_size(ax,16,False)

## 3. 3d surface plots

### measured surface coloured by slope error

In [ ]:
plot_3d_surface(mesh,slope_errors*1e3,"slope error (mrad)",fontsize=16)

### residual surface coloured by slope error

In [ ]:
dzs = (z-ideal_zs)*1e3
dz_mesh = mesher.create_mesh(x,y,dzs)

In [ ]:
fig = plt.figure()
fig.set_size_inches(10,10)
ax = fig.add_subplot(111,projection='3d')
plot_3d_surface(dz_mesh,values=slope_errors*1e3,label="slope error (mrad)",ax=ax,fontsize=16)
ax.set_zlabel("z (mm)")
plt.show()

### residual surface coloured by z residual

In [ ]:
fig = plt.figure()
fig.set_size_inches(10,10)
ax = fig.add_subplot(111,projection='3d')
plot_3d_surface(dz_mesh,values=np.mean(dz_mesh.z,axis=1),label="z residual (mm)",ax=ax,fontsize=16)
ax.set_zlabel("z (mm)")
plt.show()

## 4. check sensitivity to max aspect ratio parameter

In [ ]:
ax2_parameter = "area"
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)

In [ ]:
# max_aspect_ratios = list(np.arange(2,10,1)) + list(np.arange(10,100,10)) + list(np.arange(100,1000,100))
attempted_max_aspect_ratios = list(np.linspace(1.001,2,50)) + list(np.linspace(2.001,10,50)) + list(np.linspace(10.001,100,50))
max_aspect_ratios = []
aggregated_slope_errors = []
triangle_counts = []
mirror_areas = []
for max_aspect_ratio in attempted_max_aspect_ratios:
    print("-------------------------------------------")
    print("max_aspect_ratio = %f "%(max_aspect_ratio))
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio,"verbose":True})
    try:
        mesh = mesher.create_mesh(x, y, z)
        areas, normals, coeffs, ideal_normals, ideal_zs, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
        triangle_counts.append(len(areas))
        mirror_areas.append(sum(areas))
        aggregate = aggregator.aggregate(slope_errors,areas)
        aggregated_slope_errors.append(aggregate/np.sqrt(2))
        max_aspect_ratios.append(max_aspect_ratio)
    except Exception as e:
        print("case failed with exception: %s"%(str(e)))    

f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors,color="blue")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax.yaxis.label.set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax2 = plt.twinx(ax)
if ax2_parameter=="counts":
    ax2.plot(max_aspect_ratios,triangle_counts,color="r")
    ax2.set_ylabel("triangle counts")
elif ax2_parameter=="area":
    ax2.plot(max_aspect_ratios,mirror_areas,color="r")
    ax2.set_ylabel("mirror area (m$^2$)")
else:
    raise RuntimeError("ax2_parameter: %s not recognised"%ax2_parameter)
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()